# Bloque III — Clasificación Multiclase: Iris Dataset

**Duración estimada:** 3 horas  
**Dataset:** `Iris.csv` - Clásico dataset de clasificación con 3 especies de iris

## Objetivo de aprendizaje

El alumnado aprenderá a:
1. Entrenar modelos de clasificación multiclase (3 clases)
2. Interpretar métricas por clase (Precision, Recall, F1-score)
3. Visualizar matrices de confusión y curvas ROC multiclase
4. Comparar importancia de características entre modelos

## Las 3 especies a clasificar:
- **Iris-setosa**
- **Iris-versicolor**
- **Iris-virginica**

## Características (features):
- SepalLengthCm (Longitud del sépalo)
- SepalWidthCm (Ancho del sépalo)
- PetalLengthCm (Longitud del pétalo)
- PetalWidthCm (Ancho del pétalo)

In [ ]:
# Configuración común
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de visualización
plt.style.use('default')
sns.set_palette("husl")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

print("Librerías cargadas correctamente")

In [ ]:
# Importar librerías de scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score,
    classification_report
)
from sklearn.preprocessing import LabelBinarizer

print("Scikit-learn imports completados")
print(f"sklearn version: {__import__('sklearn').__version__}")

## 1. Carga y exploración del dataset Iris

El dataset Iris es uno de los más famosos en machine learning. Contiene 150 muestras (50 por clase) con 4 características numéricas.

In [ ]:
# Cargar el dataset Iris
import os
data_path = r'C:\temp\iris_work\Iris.csv'
print(f'Cargando datos desde: {data_path}')

df = pd.read_csv(data_path)
print(f"\nShape del dataset: {df.shape}")
print(f"Columnas: {list(df.columns)}")
df.head(10)

In [ ]:
# Información del dataset
print("="*60)
print("INFORMACION DEL DATASET")
print("="*60)

print("\nTipos de datos:")
df.info()
print("\nEstadísticas descriptivas:")
display(df.describe())

In [ ]:
# Verificar valores nulos
print("Valores nulos por columna:")
print(df.isnull().sum())
print("\n¿Hay valores duplicados?", df.duplicated().sum())

## 2. Distribución de clases y visualización

En clasificación multiclase es fundamental revisar si las clases están balanceadas. El dataset Iris está perfectamente balanceado (50 muestras por clase).

In [ ]:
# Distribución de clases
print("="*60)
print("DISTRIBUCION DE CLASES")
print("="*60)

conteo = df["Species"].value_counts()
proporcion = df["Species"].value_counts(normalize=True)

print("\nConteo absoluto:")
display(conteo)
print("\nProporcion relativa:")
display(proporcion)

# Visualizar distribución
plt.figure(figsize=(8, 5))
conteo.plot(kind="bar", color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
plt.title("Distribucion de especies de Iris", fontsize=14, fontweight='bold')
plt.xlabel("Especie", fontsize=12)
plt.ylabel("Numero de muestras", fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('C:/temp/iris_work/distribucion_clases.png')
plt.close()
print("\nGrafico guardado: distribucion_clases.png")

In [ ]:
# Visualización de las características por especie
print("="*60)
print("ANALISIS EXPLORATORIO - PAIRPLOT")
print("="*60)

features = ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']

# Pairplot para ver relaciones entre características
sns.pairplot(df, hue='Species', vars=features, diag_kind='kde', height=2.5)
plt.suptitle('Pairplot - Caracteristicas por Especie', y=1.02, fontsize=14, fontweight='bold')
plt.savefig('C:/temp/iris_work/pairplot.png')
plt.close()
print("Pairplot guardado: pairplot.png")

In [ ]:
# Boxplot para ver la distribución de cada característica
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Distribucion de caracteristicas por especie', fontsize=16, fontweight='bold')

for idx, feature in enumerate(features):
    ax = axes[idx//2, idx%2]
    df.boxplot(column=feature, by='Species', ax=ax, grid=False)
    ax.set_title(f'{feature}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Especie', fontsize=10)
    ax.set_ylabel(feature, fontsize=10)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.savefig('C:/temp/iris_work/boxplots.png')
plt.close()
print("Boxplots guardados: boxplots.png")

## 3. Preparación de datos: X (features) e y (target)

Para clasificación multiclase con 3 clases, usaremos:
- **X**: Las 4 mediciones (SepalLength, SepalWidth, PetalLength, PetalWidth)
- **y**: Species (Iris-setosa, Iris-versicolor, Iris-virginica)

Usamos estratificación para mantener la proporción de clases en train y test.

In [ ]:
# Preparar X e y
features = ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']
target = 'Species'

X = df[features]
y = df[target]

print("="*60)
print("PREPARACION DE DATOS")
print("="*60)
print(f"\nFeatures (X): {features}")
print(f"Target (y): {target}")
print(f"\nShape X: {X.shape}")
print(f"Shape y: {y.shape}")
print(f"\nClases unicas: {y.unique()}")

In [ ]:
# Codificar las etiquetas
le = LabelEncoder()
y_encoded = le.fit_transform(y)
class_names = le.classes_

print("Codificacion de etiquetas:")
for i, cls in enumerate(class_names):
    print(f"  {cls} -> {i}")

print(f"\ny codificado (primeros 10): {y_encoded[:10]}")

In [ ]:
# División train/test estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # Mantener proporción de clases
)

print("="*60)
print("DIVISION TRAIN/TEST")
print("="*60)
print(f"\nX_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

print("\nDistribucion en y_train:")
display(y_train.value_counts())
print("\nDistribucion en y_test:")
display(y_test.value_counts())

In [ ]:
# Escalar las características (importante para Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Caracteristicas escaladas con StandardScaler")
print(f"\nMedia de X_train_scaled: {np.mean(X_train_scaled, axis=0)}")
print(f"Desv. estandar de X_train_scaled: {np.std(X_train_scaled, axis=0)}")

## 4. Función de evaluación para clasificación multiclase

Calculamos métricas por clase usando One-vs-Rest (OvR) para ROC AUC en multiclase.

In [ ]:
def evaluar_clasificador_multiclase(nombre, modelo, X_train, X_test, y_train, y_test, class_names):
    """
    Evalúa un clasificador multiclase y retorna métricas.
    """
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    
    # Métricas básicas
    accuracy = accuracy_score(y_test, pred)
    
    # Precision, Recall, F1 por clase (macro average)
    precision_macro = precision_score(y_test, pred, average='macro', zero_division=0)
    recall_macro = recall_score(y_test, pred, average='macro', zero_division=0)
    f1_macro = f1_score(y_test, pred, average='macro', zero_division=0)
    
    # Precision, Recall, F1 por clase (weighted average)
    precision_weighted = precision_score(y_test, pred, average='weighted', zero_division=0)
    recall_weighted = recall_score(y_test, pred, average='weighted', zero_division=0)
    f1_weighted = f1_score(y_test, pred, average='weighted', zero_division=0)
    
    # ROC AUC multiclase (One-vs-Rest)
    if hasattr(modelo, "predict_proba"):
        proba = modelo.predict_proba(X_test)
        # Binarizar y para One-vs-Rest
        y_bin = LabelBinarizer().fit_transform(y_test)
        if y_bin.shape[1] == 2:  # Caso binario
            auc = roc_auc_score(y_test, proba[:, 1])
        else:  # Caso multiclase
            auc = roc_auc_score(y_test, proba, multi_class='ovr', average='macro')
    else:
        proba = None
        auc = np.nan
    
    return {
        "modelo": nombre,
        "accuracy": accuracy,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
        "roc_auc_macro": auc
    }, pred, proba

## 5. Modelo 1: Regresión Logística (Multinomial)

La regresión logística para multiclase en sklearn 1.8+ maneja automáticamente la clasificación multiclase usando softmax (multinomial).

In [ ]:
# Modelo 1: Logistic Regression
print("="*60)
print("ENTRENANDO MODELO 1: LOGISTIC REGRESSION")
print("="*60)

log_reg = LogisticRegression(
    solver='lbfgs',
    max_iter=1000,
    random_state=42
)

res_logreg, pred_logreg, proba_logreg = evaluar_clasificador_multiclase(
    "Logistic Regression", log_reg, 
    X_train_scaled, X_test_scaled, 
    y_train, y_test,
    class_names
)

print("\nResultados Logistic Regression:")
for key, value in res_logreg.items():
    if key != 'modelo':
        print(f"  {key}: {value:.4f}")

In [ ]:
# Reporte de clasificación detallado - Logistic Regression
print("\n" + "="*60)
print("REPORTE DE CLASIFICACION - LOGISTIC REGRESSION")
print("="*60)
print("\n", classification_report(y_test, pred_logreg, target_names=class_names))

## 6. Modelo 2: Árbol de Decisión

Los árboles de decisión son modelos interpretables que dividen el espacio de características mediante reglas if-else.

In [ ]:
# Modelo 2: Decision Tree
print("="*60)
print("ENTRENANDO MODELO 2: DECISION TREE")
print("="*60)

dt = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

res_dt, pred_dt, proba_dt = evaluar_clasificador_multiclase(
    "Decision Tree", dt,
    X_train, X_test,  # Los árboles no necesitan escalado
    y_train, y_test,
    class_names
)

print("\nResultados Decision Tree:")
for key, value in res_dt.items():
    if key != 'modelo':
        print(f"  {key}: {value:.4f}")

In [ ]:
# Reporte de clasificación detallado - Decision Tree
print("\n" + "="*60)
print("REPORTE DE CLASIFICACION - DECISION TREE")
print("="*60)
print("\n", classification_report(y_test, pred_dt, target_names=class_names))

## 7. Modelo 3: Random Forest

Random Forest es un ensemble de árboles de decisión que mejora la estabilidad y generalización mediante bagging.

In [ ]:
# Modelo 3: Random Forest
print("="*60)
print("ENTRENANDO MODELO 3: RANDOM FOREST")
print("="*60)

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    random_state=42
)

res_rf, pred_rf, proba_rf = evaluar_clasificador_multiclase(
    "Random Forest", rf,
    X_train, X_test,  # Random Forest no necesita escalado
    y_train, y_test,
    class_names
)

print("\nResultados Random Forest:")
for key, value in res_rf.items():
    if key != 'modelo':
        print(f"  {key}: {value:.4f}")

In [ ]:
# Reporte de clasificación detallado - Random Forest
print("\n" + "="*60)
print("REPORTE DE CLASIFICACION - RANDOM FOREST")
print("="*60)
print("\n", classification_report(y_test, pred_rf, target_names=class_names))

## 8. Comparación de modelos

Comparamos los tres modelos usando diferentes métricas. En clasificación balanceada, el **accuracy** es una métrica útil, pero también revisamos **F1-score macro** para no depender del desbalance.

In [ ]:
# Comparar todos los modelos
print("="*60)
print("COMPARACION DE MODELOS")
print("="*60)

df_resultados = pd.DataFrame([res_logreg, res_dt, res_rf])
df_resultados = df_resultados.round(4)

display(df_resultados)

In [ ]:
# Visualizar comparación
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Comparacion de Modelos - Metricas', fontsize=16, fontweight='bold')

metrics_to_plot = ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro']
titles = ['Accuracy', 'F1-Score (Macro)', 'Precision (Macro)', 'Recall (Macro)']

for idx, (metric, title) in enumerate(zip(metrics_to_plot, titles)):
    ax = axes[idx//2, idx%2]
    bars = ax.bar(df_resultados['modelo'], df_resultados[metric], 
                  color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Score', fontsize=10)
    ax.set_ylim([0, 1.1])
    ax.grid(axis='y', alpha=0.3)
    
    # Agregar valores encima de las barras
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('C:/temp/iris_work/comparacion_modelos.png')
plt.close()
print("Grafico comparacion guardado: comparacion_modelos.png")

## 9. Matrices de Confusión

La matriz de confusión permite ver qué clases se confunden entre sí.
- **Diagonal principal**: Predicciones correctas
- **Fuera de la diagonal**: Errores de clasificación

In [ ]:
# Matriz de confusión - Logistic Regression
print("="*60)
print("MATRIZ DE CONFUSION - LOGISTIC REGRESSION")
print("="*60)

cm_logreg = confusion_matrix(y_test, pred_logreg, labels=class_names)
disp_logreg = ConfusionMatrixDisplay(confusion_matrix=cm_logreg, display_labels=class_names)
fig, ax = plt.subplots(figsize=(8, 6))
disp_logreg.plot(cmap='Blues', values_format='d', ax=ax)
plt.title("Matriz de Confusion - Logistic Regression", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('C:/temp/iris_work/confusion_logreg.png')
plt.close()
print("Matriz guardada: confusion_logreg.png")

In [ ]:
# Matriz de confusión - Decision Tree
print("="*60)
print("MATRIZ DE CONFUSION - DECISION TREE")
print("="*60)

cm_dt = confusion_matrix(y_test, pred_dt, labels=class_names)
disp_dt = ConfusionMatrixDisplay(confusion_matrix=cm_dt, display_labels=class_names)
fig, ax = plt.subplots(figsize=(8, 6))
disp_dt.plot(cmap='Greens', values_format='d', ax=ax)
plt.title("Matriz de Confusion - Decision Tree", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('C:/temp/iris_work/confusion_dt.png')
plt.close()
print("Matriz guardada: confusion_dt.png")

In [ ]:
# Matriz de confusión - Random Forest
print("="*60)
print("MATRIZ DE CONFUSION - RANDOM FOREST")
print("="*60)

cm_rf = confusion_matrix(y_test, pred_rf, labels=class_names)
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=class_names)
fig, ax = plt.subplots(figsize=(8, 6))
disp_rf.plot(cmap='Oranges', values_format='d', ax=ax)
plt.title("Matriz de Confusion - Random Forest", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('C:/temp/iris_work/confusion_rf.png')
plt.close()
print("Matriz guardada: confusion_rf.png")

## 10. Curvas ROC y AUC (One-vs-Rest para multiclase)

Para clasificación multiclase, usamos la estrategia **One-vs-Rest**:
- Cada clase se compara contra todas las demás
- Generamos una curva ROC por clase
- El AUC macro es el promedio de los AUC individuales

In [ ]:
# Curvas ROC multiclase - Logistic Regression
print("="*60)
print("CURVAS ROC - LOGISTIC REGRESSION (One-vs-Rest)")
print("="*60)

# Binarizar las etiquetas para One-vs-Rest
lb = LabelBinarizer()
y_test_bin = lb.fit_transform(y_test)

plt.figure(figsize=(10, 8))
colors = ['blue', 'green', 'red']

for i, class_name in enumerate(class_names):
    # Calcular ROC para esta clase vs el resto
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], proba_logreg[:, i])
    auc_score = roc_auc_score(y_test_bin[:, i], proba_logreg[:, i])
    
    plt.plot(fpr, tpr, color=colors[i], lw=2,
             label=f'{class_name} (AUC = {auc_score:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Curvas ROC - Logistic Regression (One-vs-Rest)', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('C:/temp/iris_work/roc_logreg.png')
plt.close()
print("Curva ROC guardada: roc_logreg.png")

In [ ]:
# Curvas ROC multiclase - Random Forest
print("="*60)
print("CURVAS ROC - RANDOM FOREST (One-vs-Rest)")
print("="*60)

plt.figure(figsize=(10, 8))

for i, class_name in enumerate(class_names):
    # Calcular ROC para esta clase vs el resto
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], proba_rf[:, i])
    auc_score = roc_auc_score(y_test_bin[:, i], proba_rf[:, i])
    
    plt.plot(fpr, tpr, color=colors[i], lw=2,
             label=f'{class_name} (AUC = {auc_score:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Curvas ROC - Random Forest (One-vs-Rest)', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('C:/temp/iris_work/roc_rf.png')
plt.close()
print("Curva ROC guardada: roc_rf.png")

## 11. Importancia de características (Feature Importance)

Algunos modelos permiten ver qué características son más importantes para la clasificación.
- **Random Forest**: Usa la disminución del error (Gini/Entropy)
- **Logistic Regression**: Usa los coeficientes (magnitud de los pesos)

In [ ]:
# Importancia de características - Random Forest
print("="*60)
print("IMPORTANCIA DE CARACTERISTICAS - RANDOM FOREST")
print("="*60)

feature_importance = rf.feature_importances_
feature_df = pd.DataFrame({
    'feature': features,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

print("\nImportancia de características:")
display(feature_df)

# Visualizar
plt.figure(figsize=(10, 6))
bars = plt.barh(feature_df['feature'], feature_df['importance'], 
                color='#45B7D1')
plt.xlabel('Importancia', fontsize=12)
plt.ylabel('Caracteristica', fontsize=12)
plt.title('Importancia de Caracteristicas - Random Forest', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)

# Agregar valores
for bar in bars:
    width = bar.get_width()
    plt.text(width, bar.get_y() + bar.get_height()/2.,
             f'{width:.3f}', ha='left', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('C:/temp/iris_work/feature_importance_rf.png')
plt.close()
print("\nGrafico guardado: feature_importance_rf.png")

In [ ]:
# Coeficientes - Logistic Regression
print("="*60)
print("COEFICIENTES - LOGISTIC REGRESSION")
print("="*60)

# Obtener coeficientes (uno por clase vs resto)
coef = log_reg.coef_
intercept = log_reg.intercept_

for i, class_name in enumerate(class_names):
    print(f"\nCoeficientes para {class_name} vs Resto:")
    coef_df = pd.DataFrame({
        'feature': features,
        'coefficient': coef[i]
    }).sort_values('coefficient', key=abs, ascending=False)
    
    print(f"  Intercept: {intercept[i]:.4f}")
    for _, row in coef_df.iterrows():
        print(f"  {row['feature']}: {row['coefficient']:+.4f}")

In [ ]:
# Visualizar coeficientes de Logistic Regression
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Coeficientes Logistic Regression por clase', fontsize=14, fontweight='bold')

for i, class_name in enumerate(class_names):
    ax = axes[i]
    colors = ['red' if c < 0 else 'green' for c in coef[i]]
    bars = ax.barh(features, coef[i], color=colors)
    ax.set_title(f'{class_name} vs Resto', fontsize=11, fontweight='bold')
    ax.set_xlabel('Coeficiente', fontsize=9)
    ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('C:/temp/iris_work/coeficientes_logreg.png')
plt.close()
print("Grafico guardado: coeficientes_logreg.png")

## 12. Conclusiones

### Resumen de resultados:

1. **Logistic Regression (Multinomial)**:
   - Modelo lineal, altamente interpretable
   - Funciona bien en Iris porque las clases son linealmente separables
   - Da probabilidades calibradas

2. **Decision Tree**:
   - Modelo interpretable mediante reglas if-else
   - Puede sobreajustar si no se controla la profundidad
   - En Iris, suele funcionar muy bien con profundidad 3-4

3. **Random Forest**:
   - Ensemble de árboles, más robusto y generalizable
   - En este dataset pequeño, puede no superar por mucho a un árbol simple
   - Da información de importancia de características

### Hallazgos clave:
- **PetalLengthCm y PetalWidthCm** son las características más discriminativas
- **Iris-setosa** es fácilmente separable (pétalos mucho más pequeños)
- La confusión suele darse entre **Versicolor y Virginica**

### Recomendación:
Para este dataset pequeño y bien balanceado, todos los modelos funcionan bien. Si se busca interpretabilidad: **Decision Tree**. Si se busca mejor rendimiento: **Random Forest** o **Logistic Regression**.

In [ ]:
# Resumen final ejecutivo
print("="*70)
print(" "*20 + "RESUMEN EJECUTIVO" + " "*20)
print("="*70)

print("\nDATASET:")
print(f"   • Iris Dataset: 150 muestras, 4 características, 3 clases")
print(f"   • División: {X_train.shape[0]} train / {X_test.shape[0]} test")

print("\nMODELOS EVALUADOS:")
print("   1. Logistic Regression (Multinomial)")
print("   2. Decision Tree (max_depth=4)")
print("   3. Random Forest (200 árboles)")

print("\nRESULTADOS (Métricas Macro):")
for res in [res_logreg, res_dt, res_rf]:
    print(f"\n   {res['modelo']}:")
    print(f"      • Accuracy: {res['accuracy']:.3f}")
    print(f"      • Precision: {res['precision_macro']:.3f}")
    print(f"      • Recall: {res['recall_macro']:.3f}")
    print(f"      • F1-Score: {res['f1_macro']:.3f}")
    print(f"      • ROC AUC: {res['roc_auc_macro']:.3f}")

print("\nMEJOR MODELO:")
best_model = max([res_logreg, res_dt, res_rf], key=lambda x: x['f1_macro'])
print(f"   • {best_model['modelo']} con F1-Score macro = {best_model['f1_macro']:.3f}")

print("\nHALLAZGOS CLAVE:")
print("   • PetalLength y PetalWidth son las características más importantes")
print("   • Iris-setosa es fácilmente separable de las otras dos")
print("   • La confusión mayor se da entre Versicolor y Virginica")

print("\nRECOMENDACION:")
print("   • Para interpretabilidad: Decision Tree")
print("   • Para rendimiento balanceado: Random Forest")
print("   • Para probabilidades calibradas: Logistic Regression")

print("\n" + "="*70)